# ImparTail — chạy đánh giá trên Kaggle

Đọc thẳng dataset có sẵn trên Kaggle, **không tải video, không tách khung hình**:

| Dataset | Nội dung |
|---|---|
| `phmngcduy/action-genome` | annotation (`object_bbox_and_relationship.pkl`, `person_bbox.pkl`, `frame_list.txt`) |
| `phmngcduy/action-genome-frames-part-1..6` | khung hình đã tách, cấu trúc `<video>.mp4/<frame>.png` |

Tác giả có phát hành **checkpoint đã huấn luyện** nên chỉ cần chạy đánh giá.

## Ba rào cản thật và cách xử lý

1. **`torch==1.10.1+cu111`** (bản 2021) — GPU Kaggle hiện tại dùng CUDA 12, không
   có bản dựng cu111. Notebook thử torch có sẵn trước; ô chẩn đoán sẽ nói rõ nếu
   không tương thích thay vì lỗi mơ hồ.
2. **CUDA extension của fasterRCNN** (`nms`, `roi_align`, `roi_crop`) — phải biên
   dịch, và mã sinh từ thời torch 1.x thường không dựng được trên torch 2.x.
3. **`draw_rectangles`** là Cython, phải `build_ext` trước khi import.

`nms` chỉ dùng ở nhánh **sgdet**, nhưng `lib/object_detector.py:28` nạp
`fasterRCNN.resnet` ngay trong `__init__` nên **PREDCLS cũng không tránh được**.

**Đánh giá thành thật:** khả năng chạy trót lọt là thấp. Notebook này dựng để
*kiểm chứng điều đó bằng thực nghiệm* thay vì suy đoán — mỗi ô báo rõ hỏng ở đâu.

In [ ]:
# ── 0. Môi trường thật đang có gì ───────────────────────────────────
import torch, sys, subprocess, os
print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("CUDA của torch:", torch.version.cuda)
print("GPU         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ")
print("compute cap :", torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "-")
print("arch hỗ trợ :", torch.cuda.get_arch_list() if torch.cuda.is_available() else "-")
r = subprocess.run(["nvcc","--version"], capture_output=True, text=True)
print("nvcc        :", (r.stdout.strip().split("\n")[-1] if r.returncode==0 else "không có nvcc"))

In [ ]:
# ── 0b. Hạ torch cho khớp GPU ───────────────────────────────────────
# Log lần chạy trước: Kaggle cấp Tesla P100 (sm_60) nhưng cài sẵn torch 2.10
# chỉ hỗ trợ từ sm_70. Cùng lỗi đã gặp với APKGC, khắc phục bằng torch 2.4.1.
#
# ImparTail vốn viết cho torch 1.10+cu111 (2021). Không có bản cu111 nào chạy
# được trên CUDA 12, nên 2.4.1+cu121 là điểm gần nhất vừa hợp P100 vừa còn tồn
# tại. CUDA extension của fasterRCNN có dựng nổi trên API torch 2.4 hay không
# thì chưa biết — đó chính là thứ lần chạy này kiểm.
import torch, subprocess, sys
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
need = f"sm_{cap[0]}{cap[1]}" not in torch.cuda.get_arch_list() if torch.cuda.is_available() else False
print(f"GPU sm_{cap[0]}{cap[1]} | torch {torch.__version__} | cần hạ: {need}")

if need:
    !pip uninstall -y -q torchvision torchaudio 2>&1 | tail -1
    !pip install -q --force-reinstall torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -3
    r = subprocess.run([sys.executable, "-c",
        "import torch;a=torch.randn(8,8,device='cuda');"
        "print('torch',torch.__version__);print('arch',torch.cuda.get_arch_list());"
        "print('matmul OK',float((a@a).sum()))"], capture_output=True, text=True)
    print(r.stdout or r.stderr[-500:])
    print("\n>>> torch đã đổi. Các ô sau chạy qua subprocess nên nhận bản mới.")
else:
    print("không cần hạ.")

In [ ]:
# ── 1. Dataset: dò theo tệp mốc, không đoán đường dẫn ───────────────
import os, glob
ROOT = "/kaggle/input"
def find(marker, isdir=False):
    for dp, dn, fn in os.walk(ROOT):
        if (marker in dn) if isdir else (marker in fn):
            return dp
    return None

ANN = find("object_bbox_and_relationship.pkl")
print("annotation :", ANN)
if ANN: print("  ", sorted(os.listdir(ANN)))

# khung hình: thư mục cha chứa nhiều thư mục con dạng <id>.mp4
FRAME_DIRS = []
for d in sorted(glob.glob(f"{ROOT}/*")):
    for sub in (d, *glob.glob(f"{d}/*")):
        if not os.path.isdir(sub): continue
        kids = [k for k in os.listdir(sub) if k.endswith(".mp4")]
        if len(kids) > 10:
            FRAME_DIRS.append((sub, len(kids))); break
for p, n in FRAME_DIRS:
    print(f"khung hình : {p}  ({n} video)")
print(f"\nTỔNG: {sum(n for _, n in FRAME_DIRS)} thư mục video")
assert ANN and FRAME_DIRS, "thiếu dataset — gắn action-genome và action-genome-frames-part-*"

In [ ]:
# ── 2. Gộp các part khung hình thành một cây thư mục bằng symlink ───
# /kaggle/input chỉ đọc và mỗi part là một mount riêng, nên ImparTail không
# thấy được toàn bộ. Symlink rẻ hơn copy 60GB rất nhiều.
import os
AG = "/kaggle/working/data/ag"
os.makedirs(f"{AG}/frames", exist_ok=True)
os.makedirs(f"{AG}/annotations", exist_ok=True)

n = 0
for src, _ in FRAME_DIRS:
    for v in os.listdir(src):
        if not v.endswith(".mp4"): continue
        dst = f"{AG}/frames/{v}"
        if not os.path.exists(dst):
            os.symlink(f"{src}/{v}", dst); n += 1
print(f"đã liên kết {n} thư mục video -> {AG}/frames")

for f in os.listdir(ANN):
    dst = f"{AG}/annotations/{f}"
    if not os.path.exists(dst): os.symlink(f"{ANN}/{f}", dst)
print("annotations:", sorted(os.listdir(f"{AG}/annotations")))

In [ ]:
# ── 3. Mã nguồn ImparTail ───────────────────────────────────────────
%cd /kaggle/working
!rm -rf ImparTail
!git clone -q https://github.com/rohithpeddi/ImparTail.git
%cd /kaggle/working/ImparTail
import os
print("có:", [f for f in os.listdir(".") if f.endswith((".py",".sh"))][:6])

In [ ]:
# ── 4. Cython: draw_rectangles ──────────────────────────────────────
# Bắt buộc trước khi import lib.object_detector.
!pip install -q cython 2>&1 | tail -1
%cd /kaggle/working/ImparTail/lib/draw_rectangles
!python setup.py build_ext --inplace 2>&1 | tail -4
%cd /kaggle/working/ImparTail
import glob
so = glob.glob("lib/draw_rectangles/*.so")
print("build được:", so if so else "THẤT BẠI — xem log trên")

In [ ]:
# ── 5. CUDA extension của fasterRCNN ────────────────────────────────
# Chạy qua subprocess để dùng đúng bản torch vừa cài ở ô 0b.
import subprocess, sys, os
r = subprocess.run([sys.executable, "setup.py", "build", "develop"],
                   cwd="/kaggle/working/ImparTail/fasterRCNN/lib",
                   capture_output=True, text=True)
out = (r.stdout + r.stderr).strip().split("\n")
print("\n".join(out[-25:]))
print(f"\nexit code: {r.returncode}")

chk = subprocess.run([sys.executable, "-c",
       "import sys; sys.path.insert(0,'/kaggle/working/ImparTail');"
       "from fasterRCNN.lib.model.roi_layers import nms; print('nms OK')"],
      capture_output=True, text=True, cwd="/kaggle/working/ImparTail")
if "nms OK" in chk.stdout:
    print("\n✓ nms import được — rào cản chính đã gỡ")
else:
    print(f"\n✗ nms vẫn hỏng: {chk.stderr.strip().split(chr(10))[-1][:150]}")

In [ ]:
# ── 6. Thư viện còn lại ─────────────────────────────────────────────
!pip install -q torchdiffeq torchsde easydict opencv-python-headless 2>&1 | tail -2
import importlib
for m in ("torchdiffeq","torchsde","easydict","cv2"):
    try: importlib.import_module(m); print(f"  ✓ {m}")
    except Exception as e: print(f"  ✗ {m}: {e}")

In [ ]:
# ── 7. Checkpoint của tác giả ───────────────────────────────────────
# README trỏ tới UT Dallas Box. Box KHÔNG cho tải trực tiếp bằng wget/curl
# (cần phiên trình duyệt), nên phải tải tay rồi up lên Kaggle làm dataset.
#
#   VidSGG fully-supervised : https://utdallas.box.com/s/ji6aekd4d8iguqk59rfwcalhpu608mop
#   fasterRCNN đã huấn luyện: https://utdallas.box.com/s/1pspm5x8etlczoklyw4bclotsmlejagj
#
# Ô này chỉ kiểm xem đã gắn dataset checkpoint chưa.
import glob, os
ck = [p for p in glob.glob("/kaggle/input/**/*.pth", recursive=True) +
             glob.glob("/kaggle/input/**/*.tar", recursive=True)]
if ck:
    for p in ck[:8]: print(f"  {os.path.getsize(p)/1e6:8.1f} MB  {p}")
else:
    print("  chưa gắn checkpoint nào.")
    print("  Tải tay 2 link trên -> tạo dataset Kaggle -> gắn vào notebook này.")

In [ ]:
# ── 8. CHẨN ĐOÁN — chạy được tới đâu ────────────────────────────────
import os, glob, importlib, traceback
rows = []
def chk(name, fn):
    try: ok, note = fn()
    except Exception as e: ok, note = False, f"{type(e).__name__}: {str(e)[:60]}"
    rows.append((name, ok, note)); return ok

chk("khung hình đã gắn",
    lambda: (len(os.listdir(f"{AG}/frames")) > 100,
             f"{len(os.listdir(f'{AG}/frames'))} video"))
chk("annotation đã gắn",
    lambda: (os.path.exists(f"{AG}/annotations/object_bbox_and_relationship.pkl"), "pkl"))
chk("draw_rectangles (Cython)",
    lambda: (bool(glob.glob("/kaggle/working/ImparTail/lib/draw_rectangles/*.so")), "build_ext"))
def _nms():
    import sys; sys.path.insert(0, "/kaggle/working/ImparTail")
    from fasterRCNN.lib.model.roi_layers import nms
    return True, "import OK"
chk("fasterRCNN CUDA ext", _nms)
chk("checkpoint tác giả",
    lambda: (bool(glob.glob("/kaggle/input/**/*.pth", recursive=True)), "tải tay từ Box"))

print(f"  {'bước':<26} {'':3} ghi chú")
for n, ok, note in rows:
    print(f"  {n:<26} {'✓' if ok else '✗':3} {note}")

blockers = [n for n, ok, _ in rows if not ok]
print()
if blockers:
    print("CHẶN Ở:", ", ".join(blockers))
    print("\nKhông chạy được đánh giá. Đây là kết quả có giá trị — nó cho biết")
    print("chính xác rào cản nằm ở đâu, thay vì phỏng đoán.")
else:
    print("Mọi thứ sẵn sàng — chạy ô tiếp theo.")

In [ ]:
# ── 9. Đánh giá (chỉ chạy khi ô 8 báo sẵn sàng) ────────────────────
# Magic %cd / ! không dùng được bên trong khối if, nên dựng lệnh rồi
# chạy bằng subprocess.
import glob, os, subprocess, sys
ck = glob.glob("/kaggle/input/**/*.pth", recursive=True)
ready = bool(glob.glob("/kaggle/working/ImparTail/lib/draw_rectangles/*.so")) and bool(ck)

if not ready:
    print("bỏ qua — ô 8 còn rào cản chưa gỡ")
else:
    # PREDCLS rẻ nhất: cho sẵn hộp bao và nhãn vật thể, chỉ phân loại vị ngữ.
    cmd = [sys.executable, "test_sgg_methods.py",
           "--mode", "predcls", "--datasize", "large",
           "--data_path", "/kaggle/working/data/ag",
           "--model_path", ck[0]]
    print(" ".join(cmd), flush=True)
    r = subprocess.run(cmd, cwd="/kaggle/working/ImparTail",
                       capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip().split("\n")
    print("\n".join(out[-30:]))
    print(f"\nexit code: {r.returncode}")